In [ ]:
number_threshold = 31  # trong thực tế N x k = 70 x 4 = 280
max_depth = 2
ALL_SAMPLES = 150
NUM_SAMPLES_TRAIN = int(ALL_SAMPLES * 0.8)
NUM_SAMPLES_TEST  = ALL_SAMPLES - NUM_SAMPLES_TRAIN
num_label = 3
src_data = "/content/drive/MyDrive/Colab Notebooks/decision-tree-pyfhel/Release/iris_8_2.csv"
src_model_tree = "/content/drive/MyDrive/Colab Notebooks/decision-tree-pyfhel/model_tree/model_tree_ten.bin"

SOFT_STEP_COEFFICIENTS_16 = {
    5.00000000e-01,
    2.11445799e+00,
    1.15591931e-10,
    -6.38009501e+00,
    -4.91650318e-10,
    1.09534390e+01,
    8.95471329e-10,
    -1.01295272e+01,
    -8.38669100e-10,
    5.27558906e+00,
    4.36355800e-10,
    -1.54908047e+00,
    -1.27390646e-10,
    2.39094701e-01,
    1.95169389e-11,
    -1.50730122e-02,
    -1.22081599e-12
};
SOFT_STEP_COEFFICIENTS_8 = {
    0.5,
    1.23986659,
    -0.0,
    -1.05984904,
    -0.0,
    0.40068769,
    0.0,
    -0.05021892,
    0.0
};

In [ ]:
pip install pyfhel

In [ ]:
import numpy as np
from Pyfhel import Pyfhel

In [ ]:
n_mults = 11

HE = Pyfhel(key_gen=True, context_params={
    'scheme': 'CKKS',
    'n': 2**15,
    'scale': 2**40,
    'qi_sizes': [60]+ [30]*n_mults +[60]
})
print(HE)

/tmp/ipython-input-923002548.py:3: UserWarning: <Pyfhel Warning> qi_sizes [60, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 60] do not support rescaling for scale 1099511627776.0.
  HE = Pyfhel(key_gen=True, context_params={


<ckks Pyfhel obj at 0x7974924abeb0, [pk:Y, sk:Y, rtk:-, rlk:-, contx(n=32768, t=0, sec=128, qi=[60, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 60], scale=1099511627776.0, )]>


In [ ]:
HE.keyGen()
HE.relinKeyGen()
HE.rotateKeyGen()
print(HE)

<ckks Pyfhel obj at 0x7974924abeb0, [pk:Y, sk:Y, rtk:Y, rlk:Y, contx(n=32768, t=0, sec=128, qi=[60, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 60], scale=1099511627776.0, )]>


In [ ]:
!pip install pyfhel pandas numpy scikit-learn

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from Pyfhel import Pyfhel

In [ ]:
path = "/content/drive/MyDrive/Colab Notebooks/decision-tree-pyfhel/iris_8_2.csv"
df = pd.read_csv(path)

df.head()

,5.1,3.5,1.4,0.2,setosa
0,4.9,3.0,1.4,0.2,setosa
1,4.7,3.2,1.3,0.2,setosa
2,4.6,3.1,1.5,0.2,setosa
3,5.0,3.6,1.4,0.2,setosa
4,5.4,3.9,1.7,0.4,setosa


In [ ]:
X = df.iloc[:, :-1].values     # tất cả cột trừ cột cuối
labels = df.iloc[:, -1].values

In [ ]:
label_map = {
    "setosa": 0,
    "versicolor": 1,
    "virginica": 2
}

y_int = np.array([label_map[l] for l in labels])


In [ ]:
scaler = MinMaxScaler(feature_range=(-1, 1))
X = scaler.fit_transform(X)
print(X[:5])

[[-0.66666667 -0.16666667 -0.86440678 -0.91666667]
 [-0.77777778  0.         -0.89830508 -0.91666667]
 [-0.83333333 -0.08333333 -0.83050847 -0.91666667]
 [-0.61111111  0.33333333 -0.86440678 -0.91666667]
 [-0.38888889  0.58333333 -0.76271186 -0.75      ]]


In [ ]:
num_classes = len(label_map)

Y = np.zeros((len(y_int), num_classes))
Y[np.arange(len(y_int)), y_int] = 1
print(Y[:5])


[[1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]]


In [ ]:
X_train = X[:NUM_SAMPLES_TRAIN]
y_train = Y[:NUM_SAMPLES_TRAIN]
X_test = X[NUM_SAMPLES_TRAIN:]
y_test = Y[NUM_SAMPLES_TRAIN:]

In [ ]:
C_X_cols_train = []

for i in range(X_train.shape[1]): # X_train.shape[1] là số lượng đặc trưng (features) của mỗi mẫu
    col = X_train[:, i] # lấy toàn bộ giá trị của cột thứ i trong ma trận X_train
    ctxt = HE.encryptFrac(col)
    C_X_cols_train.append(ctxt)

C_Y_cols_train = []

for i in range(y_train.shape[1]):
    col = y_train[:, i]
    ctxt = HE.encryptFrac(col)
    C_Y_cols_train.append(ctxt)

C_X_cells_test = []

for i in range(X_test.shape[0]):        # từng mẫu
    row_ctxts = []
    for j in range(X_test.shape[1]):    # từng feature
        value = X_test[i, j]
        ctxt = HE.encryptFrac(np.array([value]))  # 1 ciphertext chứa 1 giá trị
        row_ctxts.append(ctxt)
    C_X_cells_test.append(row_ctxts)



In [ ]:
# Ma hoa trong so
vector_ones = np.ones(NUM_SAMPLES_TRAIN)  # kích thước 1D
C_W = HE.encryptFrac(vector_ones)

# Ma hoa nguong
min_T = -1.0
max_T = 1.0

step = (max_T - min_T - 0.4) / (number_threshold + 1)

all_thresholds = []

for i in range(1, number_threshold + 1):
    theta = min_T + i * step
    if -0.2 <= theta <= 0.2:
        continue
    if theta >= 1.0:
        break
    all_thresholds.append(theta)

number_threshold_real = len(all_thresholds)
print("Thresholds:", all_thresholds)
print("Number threshold real:", number_threshold_real)

C_thresholds = []

for t in all_thresholds:
    ctxt = HE.encryptFrac(np.array([t]))
    C_thresholds.append(ctxt)


Thresholds: [-0.95, -0.9, -0.85, -0.8, -0.75, -0.7, -0.6499999999999999, -0.6, -0.55, -0.5, -0.44999999999999996, -0.3999999999999999, -0.35, -0.29999999999999993, -0.25, 0.20000000000000018, 0.25, 0.30000000000000004, 0.3500000000000001, 0.40000000000000013, 0.4500000000000002, 0.5, 0.55]
Number threshold real: 23


In [ ]:
def multiply_ciphertexts(ct1, ct2, HE):
    """
    ct1, ct2: PyCtxt
    HE: Pyfhel object
    """
    # copy để không đổi nguyên bản
    ct1_copy = ct1.copy()
    ct2_copy = ct2.copy()

    # scale: lấy min scale
    target_scale = min(ct1_copy.scale, ct2_copy.scale)
    ct1_copy.scale = target_scale
    ct2_copy.scale = target_scale

    # Nhân và relinearize
    result = ct1_copy * ct2_copy  # Pyfhel tự handle rescale khi nhân CKKS
    result = HE.relinearize(result)
    return result

def multiply_cipher_plain(ct, pt, HE):
    """
    ct: PyCtxt
    pt: PyPtxt (Plaintext)
    HE: Pyfhel
    """
    # copy
    ct_copy = ct.copy()
    pt_copy = pt

    # scale của pt = scale của ct
    pt_copy.scale = ct_copy.scale

    # Nhân
    result = ct_copy * pt_copy
    return result

def add_ciphertexts(ct1, ct2, HE):
    ct1_copy = ct1.copy()
    ct2_copy = ct2.copy()

    # scale = min scale
    target_scale = min(ct1_copy.scale, ct2_copy.scale)
    ct1_copy.scale = target_scale
    ct2_copy.scale = target_scale

    result = ct1_copy + ct2_copy
    return result

def sub_ciphertexts(ct1, ct2, HE):
    ct1_copy = ct1.copy()
    ct2_copy = ct2.copy()

    target_scale = min(ct1_copy.scale, ct2_copy.scale)
    ct1_copy.scale = target_scale
    ct2_copy.scale = target_scale

    result = ct1_copy - ct2_copy
    return result

def sum_slots(ct, HE, n_slots):
    """
    ct: PyCtxt chứa n_slots
    return: ciphertext chứa tổng ở tất cả slot
    """
    result = ct.copy()
    for i in range(1, n_slots):
        tmp = HE.rotate(ct, i)  # rotate vector sang phải i slots
        result += tmp
    return result

def rotate_to_slot(HE, ctxt, target_index):
    """
    Rotate CKKS ciphertext so that slot[target_index] moves to slot[0]

    Parameters
    ----------
    HE : Pyfhel
        Pyfhel context (must have Galois keys)
    ctxt : PyCtxt
        Encrypted CKKS vector
    target_index : int
        steps > 0  -> rotate left
        steps < 0  -> rotate right
    """
    print(type(ctxt))
    x = HE.rotate(ctxt, target_index)
    return x

def mul_cipher_with_cipher_slot0(HE, ct, ct_ref):
    """
    Nhân ct với giá trị slot 0 của ct_ref (ct * ct_ref[0])
    Thực hiện hoàn toàn trên ciphertext (CKKS, Pyfhel)
    """

    nslots = HE.get_nSlots()

    # 1. Mask để chỉ giữ slot 0
    mask = [1.0] + [0.0] * (nslots - 1) # [1.0] + [0.0] * 2 [1.0, 0.0, 0.0]
    pt_mask = HE.encodeFrac(np.array(mask, dtype=np.float64))

    ct_slot0 = multiply_cipher_plain(ct_ref, pt_mask, HE)

    # 2. Broadcast slot 0 ra toàn bộ vector
    step = 1
    while step < nslots:
        x = rotate_to_slot(HE, ct_slot0, step)
        ct_slot0 = multiply_ciphertexts(ct_slot0, x, HE)
        step <<= 1

    out = multiply_ciphertexts(ct, ct_slot0, HE)
    # 3. Nhân với ciphertext chính
    return out




In [ ]:
def soft_step_evaluation(encrypted_z, HE, SOFT_STEP_COEFFICIENTS):
    """
    encrypted_z: PyCtxt, ciphertext của z = cx - theta
    HE: Pyfhel object
    SOFT_STEP_COEFFICIENTS: list hệ số đa thức c0, c1, ...
    """
    # 1. Tạo vector lưu các lũy thừa của z: z^1, z^2, z^4, z^8, ... (exponentiation by squaring)
    powers = [encrypted_z]  # z^1
    i = 1
    while i < len(SOFT_STEP_COEFFICIENTS):
        z_for_mul = powers[-1]
        tmp = multiply_ciphertexts(powers[-1], z_for_mul, HE)
        powers.append(tmp)
        i *= 2

    # 2. Encode c0 và khởi tạo result
    c0_plain = HE.encodeFrac([SOFT_STEP_COEFFICIENTS[0]] * HE.get_nSlots())
    result = HE.encryptFrac(c0_plain)

    # 3. Tính đa thức: result = c0 + c1*z + c2*z^2 + ...
    for i in range(1, len(SOFT_STEP_COEFFICIENTS)):
        coeff = SOFT_STEP_COEFFICIENTS[i]
        if abs(coeff) < 1e-12:
            continue

        # 3.1 Tính z^i bằng cách kết hợp các lũy thừa 2^j
        power = None
        first = True
        j = 0
        while (1 << j) <= i:
            if i & (1 << j):
                if first:
                    power = powers[j]
                    first = False
                else:
                    power = multiply_ciphertexts(power, powers[j], HE)
            j += 1

        # 3.2 Encode hệ số
        coeff_plain = HE.encodeFrac([coeff] * HE.get_nSlots())
        # 3.3 Nhân hệ số
        term = multiply_cipher_plain(power, coeff_plain, HE)
        # 3.4 Cộng vào result
        result = add_ciphertexts(result, term, HE)

    return result


In [ ]:
class NodeC:
    def __init__(self):
        self.is_leaf = False
        # Nếu là lá: list PyCtxt, mỗi ciphertext chứa trọng số cho nhãn l
        self.leaf_value_vector = []

        # Nếu không là lá: phân chia
        self.feature_index = None
        self.threshold = None    # PyCtxt

        # Node con
        self.left_child = None
        self.right_child = None

In [ ]:
def leaf_value(C_W_col, C_Y_cols, HE, num_label):
    """
    C_W_col: PyCtxt của trọng số cột W
    C_Y_cols: list PyCtxt của nhãn one-hot (column-wise)
    HE: Pyfhel object
    num_label: số nhãn
    return: list PyCtxt, mỗi element = tổng trọng số cho nhãn l
    """
    print("leaf_value()")
    C_leaf_values = []

    for l in range(num_label):
        print(f"\tProcessing label {l}")
        # Nhân C_W_col * C_Y_cols[l] (an toàn)
        P = multiply_ciphertexts(C_W_col, C_Y_cols[l], HE)

        # Nếu muốn tính tổng slots, dùng sum_slots(P, HE, n_slots)
        P = sum_slots(P, HE, HE.get_nSlots())

        C_leaf_values.append(P)

    print("Completed leaf_value()")
    return C_leaf_values


In [ ]:
def compute_weighted_counts_homo(best_feature, C_T_col, C_X_cols, C_W_col, C_Y_cols, HE,
                                 num_feature, num_label, SOFT_STEP_COEFFICIENTS):
    """
    best_feature: list PyCtxt, 1 value per feature (plaintext broadcasted to ciphertext)
    C_T_col: PyCtxt (threshold)
    C_X_cols: list PyCtxt, K features
    C_W_col: PyCtxt, trọng số
    C_Y_cols: list PyCtxt, L nhãn
    HE: Pyfhel object
    num_feature: số feature
    num_label: số nhãn
    SOFT_STEP_COEFFICIENTS: list hệ số soft-step polynomial
    """
    print("compute_weighted_counts_homo()")
    C_right_counts = [None] * num_label
    C_left_counts = [None] * num_label

    # Tính C_X[i] = sum_k(best_feature[k] * C_X_cols[k])
    print("\tTính C_X[i]")
    C_X_i = None
    first = True
    for k in range(num_feature):
        x = rotate_to_slot(HE, best_feature, k)
        term = mul_cipher_with_cipher_slot0(HE, C_X_cols[k], x)
        if C_X_i is None:
            C_X_i = term
            first = False
        else:
            C_X_i = add_ciphertexts(C_X_i, term, HE)

    # Tính Z_right = C_X[i] - C_T_col, Z_left = C_T_col - C_X[i]
    print("\tTính Z_right và Z_left")
    C_Z_right = sub_ciphertexts(C_X_i, C_T_col, HE)
    C_Z_left = sub_ciphertexts(C_T_col, C_X_i, HE)

    # Soft-step evaluation
    print("\tTính soft-step()")
    C_Phi_Right = soft_step_evaluation(C_Z_right, HE, SOFT_STEP_COEFFICIENTS)
    C_Phi_Left  = soft_step_evaluation(C_Z_left, HE, SOFT_STEP_COEFFICIENTS)

    # Nhân W * Phi
    print("\tNhân C_W_col * Phi")
    C_W_Phi_Right = multiply_ciphertexts(C_W_col, C_Phi_Right, HE)
    C_W_Phi_Left  = multiply_ciphertexts(C_W_col, C_Phi_Left, HE)

    # Tính weighted counts per label
    print("\tCompute weighted counts per label")
    for l in range(num_label):
        # RIGHT
        C_term_right = multiply_ciphertexts(C_W_Phi_Right, C_Y_cols[l], HE)
        C_right_counts[l] = sum_slots(C_term_right, HE, HE.get_nSlots())
        # LEFT
        C_term_left = multiply_ciphertexts(C_W_Phi_Left, C_Y_cols[l], HE)
        C_left_counts[l] = sum_slots(C_term_left, HE, HE.get_nSlots())

    return C_right_counts, C_left_counts

In [ ]:
def compute_gini_impurity(right_counts_, left_counts_, NUM_SAMPLES_TRAIN):
    """
    right_counts_: list of list of float, mỗi list tương ứng 1 nhãn (Lx1)
    left_counts_: list of list of float
    NUM_SAMPLES_TRAIN: tổng số mẫu train, dùng để sum slots nếu cần
    return: float, gini impurity
    """
    print("compute_gini_impurity()")
    # Sum slots nếu input là list per sample
    right_counts = [counts[0] for counts in right_counts_]
    left_counts  = [counts[0] for counts in left_counts_]

    # Tổng trọng số
    total_right = sum(right_counts)
    total_left  = sum(left_counts)
    total_all   = total_right + total_left
    if total_all < 1e-9:
        return 0.0

    # Gini bên phải
    gini_right = 0.0
    if total_right > 1e-9:
        sum_sq = sum((count / total_right)**2 for count in right_counts)
        gini_right = (1.0 - sum_sq) * (total_right / total_all)

    # Gini bên trái
    gini_left = 0.0
    if total_left > 1e-9:
        sum_sq = sum((count / total_left)**2 for count in left_counts)
        gini_left = (1.0 - sum_sq) * (total_left / total_all)

    print("Completed compute_gini_impurity()")
    return gini_right + gini_left


In [ ]:
def compute_W_phi_best(best_feature, best_threshold, C_X_cols, C_W_col, HE,
                       num_feature, SOFT_STEP_COEFFICIENTS):
    """
    best_feature: list PyCtxt one-hot
    best_threshold: PyCtxt (threshold)
    C_X_cols: list PyCtxt các feature
    C_W_col: PyCtxt trọng số
    HE: Pyfhel object
    num_feature: số feature
    SOFT_STEP_COEFFICIENTS: list hệ số soft-step polynomial
    return: tuple (C_W_new_right, C_W_new_left)
    """
    print("compute_W_phi_best()")

    # Tính X[best_feature] = sum_k(best_feature[k] * C_X_cols[k])
    print("\tTính X[best_feature]")
    C_X_i = None
    first = True
    for k in range(num_feature):
        x = rotate_to_slot(HE, best_feature, k)
        term = mul_cipher_with_cipher_slot0(HE, C_X_cols[k], x)
        if C_X_i is None:
            C_X_i = term
            first = False
        else:
            C_X_i = add_ciphertexts(C_X_i, term, HE)

    # Z_right = X - theta, Z_left = theta - X
    print("\tTính Z = X[best_feature] - theta")
    C_Z_right = sub_ciphertexts(C_X_i, best_threshold, HE)
    C_Z_left  = sub_ciphertexts(best_threshold, C_X_i, HE)

    # Soft-step evaluation
    print("\tTính soft-step(Z)")
    C_Phi_Right = soft_step_evaluation(C_Z_right, HE, SOFT_STEP_COEFFICIENTS)
    C_Phi_Left  = soft_step_evaluation(C_Z_left, HE, SOFT_STEP_COEFFICIENTS)

    # W_new = W * Phi
    print("\tTính W_new = W * Phi")
    C_W_new_right = multiply_ciphertexts(C_W_col, C_Phi_Right, HE)
    C_W_new_left  = multiply_ciphertexts(C_W_col, C_Phi_Left, HE)

    print("Completed compute_W_phi_best()")
    return C_W_new_right, C_W_new_left


In [ ]:
def train_decision_tree(C_X_cols, C_W_col, C_Y_cols, C_T_cols, depth, max_depth, HE,
                        num_feature, C_I_one_hot, NUM_SAMPLES_TRAIN, num_label,
                        SOFT_STEP_COEFFICIENTS):
    print(f"train_decision_tree() depth={depth}")
    node_c = NodeC()

    # Điều kiện dừng
    if depth >= max_depth:
        print(f"Node leaf tại depth={depth}")
        node_c.is_leaf = True
        C_leaf_values = leaf_value(C_W_col, C_Y_cols, HE, num_label)
        node_c.leaf_value_vector = C_leaf_values
        return node_c

    # Tính weighted counts cho từng feature và threshold
    print("Tính weighted counts cho từng feature và threshold")
    C_right_counts_C_T_cols_I = []
    C_left_counts_C_T_cols_I  = []

    for i in range(num_feature):
        print(f"\tFeature {i}")
        C_one_hot_feature = C_I_one_hot[i]
        C_right_counts_C_T_cols = []
        C_left_counts_C_T_cols  = []

        for j, C_T_col in enumerate(C_T_cols):
            print(f"\t\tThreshold {j}")
            C_right_counts, C_left_counts = compute_weighted_counts_homo(
                C_one_hot_feature, C_T_col, C_X_cols, C_W_col, C_Y_cols,
                HE, num_feature, num_label, SOFT_STEP_COEFFICIENTS
            )
            C_right_counts_C_T_cols.append(C_right_counts)
            C_left_counts_C_T_cols.append(C_left_counts)

        C_right_counts_C_T_cols_I.append(C_right_counts_C_T_cols)
        C_left_counts_C_T_cols_I.append(C_left_counts_C_T_cols)

    # Client decrypt và tính Gini, tìm best feature & threshold
    print("Decrypt và tính Gini, chọn best_feature & best_threshold")
    min_gini = 1e9
    best_feature_idx = -1
    best_threshold_idx = -1
    num_theta = len(C_T_cols)

    for i in range(num_feature):
        for j in range(num_theta):
            # Decrypt counts cho nhãn
            right_counts_clear = []
            left_counts_clear  = []

            for l in range(num_label):
                decoded_r = HE.decryptFrac(C_right_counts_C_T_cols_I[i][j][l])
                decoded_l = HE.decryptFrac(C_left_counts_C_T_cols_I[i][j][l])
                right_counts_clear.append(decoded_r)
                left_counts_clear.append(decoded_l)

            # Tính Gini
            current_gini = compute_gini_impurity(right_counts_clear, left_counts_clear, NUM_SAMPLES_TRAIN)
            if current_gini < min_gini:
                min_gini = current_gini
                best_feature_idx = i
                best_threshold_idx = j

    print(f"best_feature={best_feature_idx}, best_threshold={best_threshold_idx}")

    # Lấy ciphertext best_feature one-hot & best_threshold
    C_best_feature = C_I_one_hot[best_feature_idx]
    C_best_threshold = C_T_cols[best_threshold_idx]

    node_c.feature_index = C_best_feature
    node_c.threshold = C_best_threshold

    # Tính W_new cho nhánh trái & phải
    print("Tính W_new cho de quy")
    C_W_new_right, C_W_new_left = compute_W_phi_best(
        C_best_feature, C_best_threshold, C_X_cols, C_W_col,
        HE, num_feature, SOFT_STEP_COEFFICIENTS
    )

    # Recursively train children
    node_c.right_child = train_decision_tree(
        C_X_cols, C_W_new_right, C_Y_cols, C_T_cols, depth+1, max_depth,
        HE, num_feature, C_I_one_hot, NUM_SAMPLES_TRAIN, num_label, SOFT_STEP_COEFFICIENTS
    )
    node_c.left_child = train_decision_tree(
        C_X_cols, C_W_new_left, C_Y_cols, C_T_cols, depth+1, max_depth,
        HE, num_feature, C_I_one_hot, NUM_SAMPLES_TRAIN, num_label, SOFT_STEP_COEFFICIENTS
    )

    print(f"Completed train_decision_tree() at depth={depth}")
    return node_c

In [ ]:
def predict_decision_tree(node_c, C_X_cols, HE, num_feature, num_label, SOFT_STEP_COEFFICIENTS):
    print("predict_decision_tree()")

    # Nếu là node leaf
    if node_c.is_leaf:
        print("Reached leaf node. Returning leaf values.")
        return node_c.leaf_value_vector

    # Không phải leaf — tính soft-step
    i_best = node_c.feature_index      # list PyCtxt one-hot
    C_Theta = node_c.threshold         # PyCtxt

    # Tính X_i = sum_k i_best[k] * C_X_cols[k]
    C_X_i = None
    first = True
    for k in range(num_feature):
        x = rotate_to_slot(HE, i_best, k)
        term = mul_cipher_with_cipher_slot0(HE, i_best[k], x)
        if C_X_i is None:
            C_X_i = term
            first = False
        else:
            C_X_i = add_ciphertexts(C_X_i, term, HE)

    # Tính Z_right = X_i - Theta, Z_left = Theta - X_i
    C_Z_right = sub_ciphertexts(C_X_i, C_Theta, HE)
    C_Z_left  = sub_ciphertexts(C_Theta, C_X_i, HE)

    # Tính soft-step
    C_Phi_Right = soft_step_evaluation(C_Z_right, HE, SOFT_STEP_COEFFICIENTS)
    C_Phi_Left  = soft_step_evaluation(C_Z_left,  HE, SOFT_STEP_COEFFICIENTS)

    # Đệ quy
    C_Output_Right = predict_decision_tree(node_c.right_child, C_X_cols, HE, num_feature, num_label, SOFT_STEP_COEFFICIENTS)
    C_Output_Left  = predict_decision_tree(node_c.left_child,  C_X_cols, HE, num_feature, num_label, SOFT_STEP_COEFFICIENTS)

    # Nhân soft-step với output từng nhánh
    C_Out_Phi_Right = []
    C_Out_Phi_Left  = []
    for l in range(num_label):
        C_Out_Phi_Right.append(multiply_ciphertexts(C_Output_Right[l], C_Phi_Right, HE))
        C_Out_Phi_Left.append(multiply_ciphertexts(C_Output_Left[l], C_Phi_Left, HE))

    # Tổng hợp kết quả
    C_Final_Output = []
    for l in range(num_label):
        C_Final_Output.append(add_ciphertexts(C_Out_Phi_Right[l], C_Out_Phi_Left[l], HE))

    print("Completed predict_decision_tree()")
    return C_Final_Output


In [ ]:
import json
from typing import List
import numpy as np

# Hàm tính accuracy
def calculate_accuracy(decoded_predictions: List[List[float]], Y_test_onehot: List[List[float]]) -> float:
    print("calculate_accuracy()")
    NUM_LABELS = len(decoded_predictions)
    NUM_SAMPLES_TEST = len(Y_test_onehot)
    correct = 0

    for i in range(NUM_SAMPLES_TEST):
        pred_scores = [decoded_predictions[l][i] for l in range(NUM_LABELS)]
        pred_label = int(np.argmax(pred_scores))

        true_label = int(np.argmax(Y_test_onehot[i]))

        if pred_label == true_label:
            correct += 1

    acc = correct / NUM_SAMPLES_TEST
    print(f"Accuracy: {acc:.6f}")
    return acc

# Hàm tính Macro F1-Score
def calculate_f1_score(decoded_predictions: List[List[float]], Y_test_onehot: List[List[float]]) -> float:
    print("calculate_f1_score() - Macro Average")
    L = len(decoded_predictions)
    N = len(Y_test_onehot)

    true_labels = [int(np.argmax(Y_test_onehot[i])) for i in range(N)]
    predicted_labels = [int(np.argmax([decoded_predictions[l][i] for l in range(L)])) for i in range(N)]

    # Confusion Matrix
    CM = np.zeros((L, L), dtype=int)
    for t, p in zip(true_labels, predicted_labels):
        CM[t][p] += 1

    f1_scores = []
    print("\n--- F1 Score Per Class ---")
    print("Class | Precision | Recall | F1-Score")
    for l in range(L):
        TP = CM[l][l]
        FP = sum(CM[:, l]) - TP
        FN = sum(CM[l, :]) - TP
        precision = 0.0 if TP + FP == 0 else TP / (TP + FP)
        recall    = 0.0 if TP + FN == 0 else TP / (TP + FN)
        f1 = 0.0 if precision + recall < 1e-9 else 2 * (precision * recall) / (precision + recall)
        f1_scores.append(f1)
        print(f"{l:5} | {precision:9.4f} | {recall:6.4f} | {f1:8.4f}")

    macro_f1 = sum(f1_scores) / L if L > 0 else 0.0
    print(f"\nMacro F1-Score (Average): {macro_f1:.6f}")
    return macro_f1

# Hàm lưu log JSON
def save_log_json(
    filename: str,
    accuracy: float,
    NUM_SAMPLES_TRAIN: int,
    NUM_SAMPLES_TEST: int,
    src_data: str,
    number_threshold: int,
    max_depth: int,
    num_label: int,
    src_model_tree: str,
    time_encrypt: float,
    time_encrypt_thresholds: float,
    time_training: float,
    time_save_model: float,
    time_load_model: float,
    time_predict_model: float,
    start_time_ms: int,
    end_time_ms: int
) -> bool:
    try:
        with open(filename, 'r') as f:
            root = json.load(f)
            if not isinstance(root, list):
                root = []
    except:
        root = []

    entry = {
        "accuracy": accuracy,
        "NUM_SAMPLES_TRAIN": NUM_SAMPLES_TRAIN,
        "NUM_SAMPLES_TEST": NUM_SAMPLES_TEST,
        "src_data": src_data,
        "number_threshold": number_threshold,
        "max_depth": max_depth,
        "num_label": num_label,
        "src_model_tree": src_model_tree,
        "time_encrypt": time_encrypt,
        "time_encrypt_thresholds": time_encrypt_thresholds,
        "time_training": time_training,
        "time_save_model": time_save_model,
        "time_load_model": time_load_model,
        "time_predict_model": time_predict_model,
        "start_time_ms": start_time_ms,
        "end_time_ms": end_time_ms
    }
    root.append(entry)

    try:
        with open(filename, 'w') as f:
            json.dump(root, f, indent=4)
        print(f"Log saved to {filename}")
        return True
    except:
        print("Cannot open model file to save!")
        return False

# Hàm in cây đã giải mã
def print_tree_decrypted(node_c, HE, depth=0):
    if node_c is None:
        return

    indent = "  " * depth

    if node_c.is_leaf:
        print(indent + "Leaf Node: [", end="")
        vals = []
        for ctxt in node_c.leaf_value_vector:
            pt = HE.decryptFrac(ctxt)
            vals.append(sum(pt))
        print(", ".join(f"{v:.4f}" for v in vals) + "]")
        return

    # Internal node: decode feature_index one-hot -> index
    feature_index = -1
    for i, ctxt in enumerate(node_c.feature_index):
        val = HE.decryptFrac(ctxt)[0]
        if val > 0.5:
            feature_index = i
            break

    threshold = HE.decryptFrac(node_c.threshold)[0]
    print(indent + f"Internal Node: Feature Index = {feature_index}, Threshold = {threshold:.4f}")

    print_tree_decrypted(node_c.left_child, HE, depth+1)
    print_tree_decrypted(node_c.right_child, HE, depth+1)


In [ ]:
import time

num_feature = X_train.shape[1]
C_I_one_hot = []
for i in range(num_feature):
    vec = np.zeros(num_feature)
    vec[i] = 1.0
    x = HE.encryptFrac(vec)
    C_I_one_hot.append(x)

In [ ]:
start_time_ms = time.time()

# Train tree
root_node = train_decision_tree(
    C_X_cols=C_X_cols_train,
    C_W_col=C_W,
    C_Y_cols=C_Y_cols_train,
    C_T_cols=C_thresholds,
    depth=0,
    max_depth=max_depth,
    HE=HE,
    num_feature=num_feature,
    C_I_one_hot=C_I_one_hot,
    NUM_SAMPLES_TRAIN=NUM_SAMPLES_TRAIN,
    num_label=num_label,
    SOFT_STEP_COEFFICIENTS=SOFT_STEP_COEFFICIENTS_16
)

# ===================== Predict =====================
decoded_predictions = []

for i in range(len(C_X_cells_test)):
    C_X_i_sample = C_X_cells_test[i]
    C_pred = predict_decision_tree(
        node_c=root_node,
        C_X_cols=C_X_i_sample,
        HE=HE,
        num_feature=num_feature,
        num_label=num_label,
        SOFT_STEP_COEFFICIENTS=SOFT_STEP_COEFFICIENTS_8
    )
    # Decode output
    decoded = [HE.decryptFrac(ctxt) for ctxt in C_pred]
    decoded_predictions.append(decoded)

# Transpose to L x N for compatibility
decoded_predictions = np.array(decoded_predictions).T.tolist()

# ===================== Accuracy & F1 =====================
accuracy = calculate_accuracy(decoded_predictions, y_test.tolist())
f1_macro = calculate_f1_score(decoded_predictions, y_test.tolist())

# ===================== Print tree =====================
print_tree_decrypted(root_node, HE)

# ===================== Save log JSON =====================
save_log_json(
    filename="/content/drive/MyDrive/Colab Notebooks/decision-tree-pyfhel/he_tree_log.json",
    accuracy=accuracy,
    NUM_SAMPLES_TRAIN=NUM_SAMPLES_TRAIN,
    NUM_SAMPLES_TEST=NUM_SAMPLES_TEST,
    src_data=src_data,
    number_threshold=number_threshold,
    max_depth=max_depth,
    num_label=num_label,
    src_model_tree=src_model_tree,
    time_encrypt=0.0,
    time_encrypt_thresholds=0.0,
    time_training=0.0,
    time_save_model=0.0,
    time_load_model=0.0,
    time_predict_model=0.0,
    start_time_ms=0,
    end_time_ms=0
)

train_decision_tree() depth=0
Tính weighted counts cho từng feature và threshold
	Feature 0
		Threshold 0
compute_weighted_counts_homo()
	Tính C_X[i]
<class 'Pyfhel.PyCtxt.PyCtxt'>
<class 'Pyfhel.PyCtxt.PyCtxt'>
<class 'NoneType'>


TypeError: std::bad_cast